# 9.3 — Debug de Queries

Notebook para inspecionar e depurar o pipeline de consulta RAG.
- Carrega `src/06_query.py` via importlib (nome começa com dígito)
- Requer Qdrant indexado em `qdrant_db/` e modelos bge-m3 disponíveis

> **Nota**: O carregamento dos modelos (bge-m3 embedding + bge-reranker) pode levar
> alguns minutos na primeira execução. O bge-m3 requer ~4 GB de RAM (CPU) ou GPU.

In [ ]:
import sys
import importlib.util
import logging
from pathlib import Path

import matplotlib.pyplot as plt

# Adiciona raiz do projeto ao path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

# Configura logging para ver mensagens de debug do pipeline
logging.basicConfig(
    level=logging.DEBUG,
    format='%(levelname)s | %(name)s | %(message)s'
)

print('ROOT:', ROOT)

In [ ]:
# Inicializa QdrantClient
# Precisa que src/05_embed_index.py tenha sido executado
try:
    from qdrant_client import QdrantClient
    from src.config import QDRANT_PATH, COLLECTION_NAME

    client = QdrantClient(path=str(QDRANT_PATH))
    col_info = client.get_collection(COLLECTION_NAME)
    print(f'Collection: {COLLECTION_NAME}')
    print(f'Vetores indexados: {col_info.points_count:,}')
    print(f'Status: {col_info.status}')
except Exception as e:
    print(f'Erro ao conectar ao Qdrant: {e}')
    print('Execute src/05_embed_index.py para criar o indice.')
    client = None

## Teste de pergunta

In [ ]:
# Carrega o modulo 06_query via importlib (nome com digito nao permite import direto)
# AVISO: embed_query carrega bge-m3 (~4 GB RAM) na primeira chamada
try:
    spec = importlib.util.spec_from_file_location(
        'query_module',
        ROOT / 'src' / '06_query.py'
    )
    query_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(query_mod)

    process_query          = query_mod.process_query
    embed_query            = query_mod.embed_query
    hybrid_search          = query_mod.hybrid_search
    reciprocal_rank_fusion = query_mod.reciprocal_rank_fusion
    lookup_parents         = query_mod.lookup_parents
    rerank                 = query_mod.rerank
    format_context         = query_mod.format_context
    query_pipeline         = query_mod.query_pipeline

    print('Modulo 06_query carregado com sucesso.')
except Exception as e:
    print(f'Erro ao carregar 06_query: {e}')
    raise

In [ ]:
# Define a pergunta de teste e roda o pipeline completo
# use_rerank=False: pula o cross-encoder para ser mais rapido neste debug inicial
# AVISO: primeira chamada vai carregar bge-m3 (pode demorar 2-5 minutos)

QUESTION = "O que e microgeracao distribuida?"

if client is None:
    print('Cliente Qdrant nao disponivel. Crie o indice primeiro.')
else:
    print(f'Pergunta: {QUESTION}')
    print('Executando pipeline (sem reranking)...')

    result = query_pipeline(
        question=QUESTION,
        client=client,
        debug=True,
        use_rerank=False,
    )
    print('Pipeline concluido.')

In [ ]:
# Mostra filtros extraidos e query limpa
if client is not None and 'result' in dir():
    processed = process_query(QUESTION)
    print('=== Query Processing ===')
    print(f'Original : {processed["original"]}')
    print(f'Limpa    : {processed["clean"]}')
    print(f'Filtros  : {processed["filters"]}')
    print()
    print('=== Timing (segundos) ===')
    for etapa, t in result.get('timing', {}).items():
        print(f'  {etapa:<20}: {t:.3f}s')
else:
    print('result nao disponivel.')

In [ ]:
# Top-5 contextos recuperados com score RRF
if client is not None and 'result' in dir():
    contexts = result.get('contexts', [])
    print(f'Contextos recuperados: {len(contexts)}')
    print()

    for i, ctx in enumerate(contexts[:5], 1):
        print(f'--- Contexto #{i} ---')
        print(f'  doc_id     : {ctx.get("doc_id", "N/A")}')
        print(f'  tipo_sigla : {ctx.get("tipo_sigla", "N/A")}')
        print(f'  numero     : {ctx.get("numero", "N/A")}')
        print(f'  ano        : {ctx.get("ano", "N/A")}')
        print(f'  chunk_type : {ctx.get("chunk_type", "N/A")}')
        print(f'  rrf_score  : {ctx.get("rrf_score", 0):.4f}')
        texto = ctx.get('texto_parent') or ctx.get('texto', '')
        print(f'  texto      : {str(texto)[:300]}...')
        print()
else:
    print('result nao disponivel.')

In [ ]:
# Resposta final gerada pelo LLM
if client is not None and 'result' in dir():
    print('=== Resposta Final ===')
    print(result.get('answer', '(sem resposta)'))
else:
    print('result nao disponivel.')

## Comparação: com e sem reranking

In [ ]:
# Descomente para comparar resultados com e sem reranking
# AVISO: use_rerank=True carrega bge-reranker-v2-m3 (~1 GB adicional)

# if client is not None:
#     import time
#
#     QUESTION_COMP = "O que e microgeracao distribuida?"
#
#     print('--- SEM reranking ---')
#     t0 = time.time()
#     res_norerank = query_pipeline(QUESTION_COMP, client=client, debug=False, use_rerank=False)
#     print(f'Tempo total: {time.time()-t0:.1f}s')
#     print('Top-1 contexto:', res_norerank['contexts'][0]['doc_id'] if res_norerank['contexts'] else 'N/A')
#
#     print()
#     print('--- COM reranking ---')
#     t0 = time.time()
#     res_rerank = query_pipeline(QUESTION_COMP, client=client, debug=False, use_rerank=True)
#     print(f'Tempo total: {time.time()-t0:.1f}s')
#     print('Top-1 contexto:', res_rerank['contexts'][0]['doc_id'] if res_rerank['contexts'] else 'N/A')
#
#     print()
#     print('=== Resposta SEM reranking ===')
#     print(res_norerank['answer'])
#     print()
#     print('=== Resposta COM reranking ===')
#     print(res_rerank['answer'])

print('Celula comentada. Descomente para executar a comparacao.')